# TKAN v2.5 — Signal & Volatility Playground

Interactive exploration of **volatility regimes**, **neural network predictions**, and **trading signals** on **LVC** (Lyxor CAC 40 x2 Leveraged ETF).

Uses pre-trained walk-forward TKAN models (23 cycles, 2015–2026) from `train_models.py`.

**Data sources**: Sfera DB (CAC OHLCV + IVol) with CSV fallback, Yahoo Finance (LVC).

**Sections**
1. Setup & Data Loading
2. Volatility Regime Charts (IVol, RVol, spread)
3. Neural Network Signal Generation
4. Signal Overlay Charts
5. Signal-Based Backtest
6. Interactive Parameter Tuning
7. Performance Metrics

## 1. Setup & Data Loading

In [7]:
import os, json, pickle, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
from signum import Chart, Dashboard
from signum.engine.statchart import StatChart

# Keep Plotly only for statistical charts (distributions, scatter, heatmaps)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook'

import tensorflow as tf
from tkan import TKAN
from sklearn.preprocessing import RobustScaler
import psycopg
import yfinance as yf
from datetime import datetime, timedelta

# Paths — notebook lives in Research/v2/
SCRIPT_DIR  = os.path.dirname(os.path.abspath("__file__"))
ROOT_DIR    = os.path.dirname(os.path.dirname(SCRIPT_DIR))
DATA_DIR    = os.path.join(ROOT_DIR, "Data")
WEIGHTS_DIR = os.path.join(SCRIPT_DIR, "weights")
OUTPUT_DIR  = os.path.join(SCRIPT_DIR, "Output")

# Sfera DB config (up 3 levels from ROOT_DIR to reach "Business & Investments")
SFERA_ENV = os.path.normpath(os.path.join(
    ROOT_DIR, "..", "..", "..", "RU Market Data", "Python", "Sfera", ".env"
))

# Config (must match train_models.py)
WINDOW_SIZE      = 10
PREDICTION_DAYS  = 10
BACKTEST_START   = '2015-01-01'
RETRAIN_FREQ     = 126
IVOL_EXIT_WINDOW = 126
DROPOUT_RATE     = 0.2
INITIAL_CAPITAL  = 100_000
MC_SAMPLES       = 50

FEATURE_COLS = [
    'log_return_1d', 'high_low_range', 'close_to_high',
    'volume_ratio', 'close_vs_sma15',
    'ivol_zscore', 'ivol_ema_ratio', 'ivol_pctl', 'ivol_roc5',
    'rvol_park20_zscore', 'vol_spread',
    'return_5d', 'return_20d',
]

In [8]:
# ── Sfera DB helpers ──────────────────────────────────────────────────
def _sfera_connstr():
    if not os.path.isfile(SFERA_ENV):
        return None
    cfg = {}
    with open(SFERA_ENV, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                cfg[k.strip()] = v.strip()
    return (f"host={cfg.get('DB_HOST','localhost')} port={cfg.get('DB_PORT','5432')} "
            f"dbname={cfg.get('DB_NAME','sfera')} user={cfg.get('DB_USER','postgres')} "
            f"password={cfg.get('DB_PASSWORD','')}")

def _load_from_db(query, label):
    connstr = _sfera_connstr()
    if connstr is None:
        return None
    try:
        with psycopg.connect(connstr) as conn:
            df = pd.read_sql(query, conn, parse_dates=['date'], index_col='date')
        print(f"  {label} from Sfera DB: {df.index[0].date()} -> {df.index[-1].date()} ({len(df)} rows)")
        return df
    except Exception as e:
        print(f"  Sfera DB ({label}) failed: {e}")
        return None

def _load_or_fallback(query, label, csv_path):
    df = _load_from_db(query, label)
    if df is None or df.empty:
        df = pd.read_csv(csv_path, parse_dates=['date'], index_col='date').sort_index()
        print(f"  {label} from CSV: {df.index[0].date()} -> {df.index[-1].date()}")
    return df

def update_lvc_from_yfinance(lvc_path, ticker="LVC.PA"):
    lvc = pd.read_csv(lvc_path, parse_dates=['date'], index_col='date').sort_index()
    last_date = lvc.index[-1]
    yf_data = yf.download(ticker, start=(last_date + timedelta(days=1)).strftime('%Y-%m-%d'),
                          end=datetime.now().strftime('%Y-%m-%d'), progress=False, auto_adjust=True)
    if yf_data.empty:
        print(f"  LVC up to date ({last_date.date()})")
        return lvc
    if yf_data.columns.nlevels > 1:
        yf_data.columns = yf_data.columns.droplevel(1)
    yf_data.index.name = 'date'
    yf_data.columns = [c.lower() for c in yf_data.columns]
    yf_data = yf_data[['open', 'high', 'low', 'close', 'volume']]
    combined = pd.concat([lvc, yf_data[~yf_data.index.isin(lvc.index)]]).sort_index()
    combined.to_csv(lvc_path)
    print(f"  LVC updated: {last_date.date()} -> {combined.index[-1].date()} (+{len(combined)-len(lvc)} rows)")
    return combined

# ── Load & update data ───────────────────────────────────────────────
print("Loading data...")

lvc = update_lvc_from_yfinance(os.path.join(DATA_DIR, "lvc_ohlcv.csv"))

cac = _load_or_fallback(
    "SELECT trade_date AS date, open_price AS open, high_price AS high, "
    "low_price AS low, close_price AS close, volume "
    "FROM bbgidx.index_prices WHERE ticker = 'CAC' ORDER BY trade_date",
    "CAC OHLCV", os.path.join(DATA_DIR, "cac_ohlcv.csv"))

ivol_raw = _load_or_fallback(
    'SELECT trade_date AS date, "3m_50d_ivol" AS ivol '
    "FROM bbgidx.index_implied_vol WHERE ticker = 'CAC' ORDER BY trade_date",
    "CAC IVol", os.path.join(DATA_DIR, "cac_ivol.csv"))

# Align on common dates
common = lvc.index.intersection(ivol_raw.index).intersection(cac.index)
df = pd.DataFrame({
    'open': lvc.loc[common, 'open'], 'high': lvc.loc[common, 'high'],
    'low': lvc.loc[common, 'low'],   'close': lvc.loc[common, 'close'],
    'volume': lvc.loc[common, 'volume'],
    'ivol': ivol_raw.loc[common, 'ivol'],
    'cac_open': cac.loc[common, 'open'], 'cac_high': cac.loc[common, 'high'],
    'cac_low': cac.loc[common, 'low'],   'cac_close': cac.loc[common, 'close'],
}).sort_index()

print(f"\nLVC data: {df.index[0].date()} to {df.index[-1].date()} ({len(df)} rows)")
print(f"Columns: {list(df.columns)}")
df.tail(3)

Loading data...


$LVC.PA: possibly delisted; no price data found  (1d 2026-03-20 -> 2026-03-20)

1 Failed download:
['LVC.PA']: possibly delisted; no price data found  (1d 2026-03-20 -> 2026-03-20)


  LVC up to date (2026-03-19)
  CAC OHLCV from Sfera DB: 2000-01-03 -> 2026-03-18 (6702 rows)
  CAC IVol from Sfera DB: 2007-01-02 -> 2026-03-18 (4916 rows)

LVC data: 2008-05-23 to 2026-03-18 (4558 rows)
Columns: ['open', 'high', 'low', 'close', 'volume', 'ivol', 'cac_open', 'cac_high', 'cac_low', 'cac_close']


,open,high,low,close,volume,ivol,cac_open,cac_high,cac_low,cac_close
date,,,,,,,,,,
2026-03-16,40.540001,41.040001,39.900002,40.715000,243491,19.8836,7921.020020,7968.649902,7856.609863,7935.970215
2026-03-17,40.619999,41.590000,40.575001,41.130001,148172,19.4748,7919.870117,8022.270020,7919.870117,7974.490234
2026-03-18,41.599998,42.044998,40.794998,41.064999,234514,18.7715,8008.930176,8048.790039,8008.930176,8048.640137


In [9]:
# Build all features (identical to train_models.py)
df['log_return_1d']  = np.log(df['close'] / df['close'].shift(1))
df['high_low_range'] = np.log(df['high'] / df['low'])
df['close_to_high']  = np.log(df['close'] / df['high'])

vol_ma20 = df['volume'].rolling(20).mean()
df['volume_ratio']   = df['volume'] / vol_ma20

sma15 = df['close'].rolling(15, min_periods=1).mean()
df['close_vs_sma15'] = np.log(df['close'] / sma15)

ivol_m63 = df['ivol'].rolling(63).mean()
ivol_s63 = df['ivol'].rolling(63).std()
df['ivol_zscore']    = (df['ivol'] - ivol_m63) / ivol_s63
ivol_ema20           = df['ivol'].ewm(span=20).mean()
df['ivol_ema_ratio'] = df['ivol'] / ivol_ema20
df['ivol_pctl']      = df['ivol'].rolling(IVOL_EXIT_WINDOW, min_periods=20).rank(pct=True)
df['ivol_roc5']      = df['ivol'].pct_change(5)

# Parkinson realized volatility on CAC underlying
log_hl      = np.log(df['cac_high'] / df['cac_low'])
park_factor = 1.0 / (4.0 * np.log(2))
rvol_park20 = np.sqrt(park_factor * (log_hl ** 2).rolling(20).mean() * 252) * 100
rvol_m63    = rvol_park20.rolling(63).mean()
rvol_s63    = rvol_park20.rolling(63).std()
df['rvol_park20']        = rvol_park20
df['rvol_park20_zscore'] = (rvol_park20 - rvol_m63) / rvol_s63
df['vol_spread']         = df['ivol'] - rvol_park20
df['return_5d']          = np.log(df['close'] / df['close'].shift(5))
df['return_20d']         = np.log(df['close'] / df['close'].shift(20))

# Extra derived features for exploration
df['ivol_sma20']   = df['ivol'].rolling(20).mean()
df['ivol_sma63']   = ivol_m63
df['ivol_ewma20']  = ivol_ema20
df['rvol_park10']  = np.sqrt(park_factor * (log_hl ** 2).rolling(10).mean() * 252) * 100
df['vol_spread10'] = df['ivol'] - df['rvol_park10']

# SMA crosses for momentum
df['sma20']  = df['close'].rolling(20).mean()
df['sma50']  = df['close'].rolling(50).mean()
df['sma100'] = df['close'].rolling(100).mean()

# Lag features for model input (predict tomorrow using today's info)
lagged = {col: df[col].shift(1) for col in FEATURE_COLS}
X = pd.DataFrame(lagged, index=df.index)
y = df['close']
df.dropna(inplace=True)
X = X.loc[df.index]; y = y.loc[df.index]
mask = X.notna().all(axis=1)
X = X.loc[mask]; y = y.loc[mask]

print(f"Feature matrix: {X.shape}  (13 stationary features, lagged 1d)")
print(f"Clean data: {df.index[0].date()} to {df.index[-1].date()}")
X.describe().round(4)

Feature matrix: (4400, 13)  (13 stationary features, lagged 1d)
Clean data: 2009-01-02 to 2026-03-18


,log_return_1d,high_low_range,close_to_high,volume_ratio,close_vs_sma15,ivol_zscore,ivol_ema_ratio,ivol_pctl,ivol_roc5,rvol_park20_zscore,vol_spread,return_5d,return_20d
count,4400.0000,4400.0000,4400.0000,4400.0000,4400.0000,4400.0000,4400.0000,4400.0000,4400.0000,4400.0000,4400.0000,4400.0000,4400.0000
mean,0.0004,0.0270,-0.0130,1.0243,0.0021,-0.1353,0.9978,0.4307,0.0037,-0.0754,4.9152,0.0021,0.0084
std,0.0246,0.0174,0.0142,0.6422,0.0523,1.2556,0.0815,0.3227,0.0972,1.3461,3.3061,0.0554,0.1058
min,-0.2861,0.0000,-0.1830,0.0000,-0.5619,-2.8213,0.7771,0.0079,-0.3206,-3.8624,-18.1795,-0.5990,-1.0191
25%,-0.0109,0.0156,-0.0171,0.6389,-0.0210,-1.0818,0.9477,0.1270,-0.0543,-1.0294,3.4603,-0.0231,-0.0429
50%,0.0015,0.0227,-0.0086,0.9102,0.0084,-0.3480,0.9847,0.3889,-0.0073,-0.2968,5.0742,0.0063,0.0160
75%,0.0130,0.0329,-0.0038,1.2493,0.0335,0.5715,1.0328,0.7143,0.0490,0.8353,6.7583,0.0324,0.0714
max,0.1400,0.2041,0.0149,20.0000,0.2055,5.8509,1.9082,1.0000,1.1123,5.0255,18.2995,0.3179,0.3877


## 2. Volatility Regime Charts

Deep dive into CAC implied volatility, realized volatility, and the vol spread.  
Look for regime transitions that could drive entry/exit signals.

In [10]:
# ── 2a. IVol + RVol + Vol Spread (4-pane Signum dashboard) ───────────
bt_start = pd.Timestamp(BACKTEST_START)
dbt = df.loc[bt_start:]

# Helper: build a time-indexed DataFrame for Signum line() calls
def ts(series, name='value'):
    s = series.dropna()
    return pd.DataFrame({'time': s.index.strftime('%Y-%m-%d'), name: s.values})

# Pane 1: LVC price + SMAs
p1 = Chart(height=300, watermark='LVC Close')
p1.line(ts(dbt['close'], 'value'), name='LVC Close', color='#4169E1')
p1.line(ts(dbt['sma20'], 'value'), name='SMA 20', color='#FFA500', width=1)
p1.line(ts(dbt['sma50'], 'value'), name='SMA 50', color='#228B22', width=1)

# Pane 2: IVol vs RVol
p2 = Chart(height=220, watermark='IVol vs RVol')
p2.line(ts(dbt['ivol'], 'value'), name='CAC IVol (3M 50D)', color='#FFA500')
p2.line(ts(dbt['rvol_park20'], 'value'), name='RVol Parkinson 20d', color='#1E90FF', width=1)
p2.line(ts(dbt['ivol_sma63'], 'value'), name='IVol SMA 63', color='#FF0000', width=1)

# Pane 3: Vol spread (baseline at 0 — green above, red below)
p3 = Chart(height=180, watermark='Vol Spread')
p3.baseline(ts(dbt['vol_spread'], 'value'), base_value=0)

# Pane 4: IVol percentile with gate lines
p4 = Chart(height=200, watermark='IVol Percentile')
p4.line(ts(dbt['ivol_pctl'], 'value'), name='IVol Pctl', color='#FF8C00')
p4.price_line(0.80, title='80% gate', color='#FF0000')
p4.price_line(0.20, title='Low vol', color='#228B22')

# Build high-vol shading DataFrame
shade_df = pd.DataFrame({
    'time': dbt.index.strftime('%Y-%m-%d'),
    'position': (dbt['ivol_pctl'] > 0.80).astype(int).values
})
p1.shade(shade_df, color='#FF0000', opacity=0.06)

dash_vol = Dashboard(panes=[p1, p2, p3, p4],
    titles=['LVC Close', 'CAC Implied Vol vs Realized Vol',
            'Vol Spread (IVol \u2212 RVol)', 'IVol Percentile (126d rolling)'])
dash_vol.show()

In [11]:
# ── 2b. CAC 3M Implied Volatility distribution ──────────────────────
(StatChart(theme="distfit", height=380,
           title="CAC 3M 50D Implied Volatility \u2014 Distribution (2015\u2013present)")
    .distribution(dbt['ivol'].dropna().values,
                  name="CAC IVol (3M 50D)", bins=80, fit=True,
                  percentiles=[25, 50, 75, 80, 90, 95, 99])
)

In [12]:
# ── 2c. Rolling correlation: IVol vs LVC returns ─────────────────────
dbt_corr = dbt[['log_return_1d', 'ivol', 'rvol_park20', 'vol_spread']].dropna()
roll_corr_ivol = dbt_corr['log_return_1d'].rolling(63).corr(dbt_corr['ivol'])
roll_corr_rvol = dbt_corr['log_return_1d'].rolling(63).corr(dbt_corr['rvol_park20'])
roll_corr_spread = dbt_corr['log_return_1d'].rolling(63).corr(dbt_corr['vol_spread'])

p_c1 = Chart(height=250, watermark='LVC Close')
p_c1.line(ts(dbt['close'], 'value'), name='LVC', color='#4169E1')

p_c2 = Chart(height=300, watermark='63d Rolling Corr')
p_c2.line(ts(roll_corr_ivol, 'value'), name='IVol \u2194 Return', color='#FFA500')
p_c2.line(ts(roll_corr_rvol, 'value'), name='RVol \u2194 Return', color='#1E90FF', width=1)
p_c2.line(ts(roll_corr_spread, 'value'), name='Vol Spread \u2194 Return', color='#800080', width=1)
p_c2.price_line(0.0, title='zero', color='#808080')

Dashboard(panes=[p_c1, p_c2],
    titles=['LVC Close', '63-day Rolling Correlation with LVC Returns']).show()

## 3. Load Pre-Trained Neural Network

Load walk-forward TKAN models (23 cycles). Build helper functions for single-pass and MC Dropout predictions.

In [13]:
# Load manifest & build model helpers
with open(os.path.join(WEIGHTS_DIR, "manifest.json")) as f:
    manifest = json.load(f)

cycles = sorted(manifest['cycles'].items(), key=lambda x: x[1]['cycle_num'])
print(f"Trained cycles: {len(cycles)}  (config hash: {manifest['config_hash']})")
print(f"First: {cycles[0][0]}  Last: {cycles[-1][0]}")

def build_model(n_features):
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(shape=(WINDOW_SIZE, n_features)),
        TKAN(100, return_sequences=True, use_bias=True),
        tf.keras.layers.Dropout(DROPOUT_RATE),
        TKAN(100, return_sequences=True, use_bias=True),
        tf.keras.layers.Dropout(DROPOUT_RATE),
        TKAN(100, return_sequences=True, use_bias=True),
        tf.keras.layers.Dropout(DROPOUT_RATE),
        tf.keras.layers.Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

def load_cycle(cycle_key, n_features):
    model = build_model(n_features)
    model.load_weights(os.path.join(WEIGHTS_DIR, f"{cycle_key}.weights.h5"))
    with open(os.path.join(WEIGHTS_DIR, f"{cycle_key}_scaler_X.pkl"), 'rb') as f:
        scaler_X = pickle.load(f)
    return model, scaler_X

def predict_single(model, scaler_X, X_df, idx, anchor_price):
    """Single forward pass \u2014 returns predicted EUR prices for next WINDOW_SIZE days."""
    X_win = X_df.iloc[idx - WINDOW_SIZE:idx]
    X_sc = scaler_X.transform(X_win).reshape(1, WINDOW_SIZE, -1)
    preds_ratio = model(X_sc, training=False).numpy().squeeze()
    return preds_ratio * anchor_price

def predict_mc(model, scaler_X, X_df, idx, anchor_price, n_samples=MC_SAMPLES):
    """MC Dropout: returns (mean_prices, lo_95, hi_95) in EUR."""
    X_win = X_df.iloc[idx - WINDOW_SIZE:idx]
    X_sc = scaler_X.transform(X_win).reshape(1, WINDOW_SIZE, -1)
    X_batch = np.repeat(X_sc, n_samples, axis=0)
    preds_ratio = model(X_batch, training=True).numpy().squeeze(-1)
    preds_eur = preds_ratio * anchor_price
    m = preds_eur.mean(axis=0)
    s = preds_eur.std(axis=0)
    return m, m - 1.96 * s, m + 1.96 * s

# Pre-compute cycle boundaries
cycle_boundaries = [(info['train_end_idx'], key) for key, info in cycles]
model_cache = {}
n_features = X.shape[1]
dates = X.index
bt_start_idx = dates.searchsorted(bt_start)

print(f"\nBacktest starts at index {bt_start_idx} ({dates[bt_start_idx].date()})")
print(f"Model helpers ready.")

Trained cycles: 23  (config hash: ec17dfc31438)
First: cycle_001_2015-01-02  Last: cycle_023_2025-10-30

Backtest starts at index 1531 (2015-01-02)
Model helpers ready.


## 4. Generate Rule-Based Volatility Signals

Create entry/exit signals from volatility features:
- **IVol regime gate**: block entries when IVol percentile > threshold
- **Vol spread crossover**: buy when spread crosses below zero (RVol > IVol \u2192 mean reversion expected)
- **IVol momentum**: buy when IVol is declining (negative RoC) and below its EMA

In [14]:
# ── 4a. Rule-based signal generation ─────────────────────────────────
# Tunable parameters (play with these!)
IVOL_PCTL_GATE   = 0.80   # block entries above this percentile
IVOL_PCTL_ENTRY  = 0.50   # enter when IVol pctl drops below this (calm regime)
SPREAD_CROSS     = 0.0    # vol spread crossing below this \u2192 potential entry
IVOL_ROC_THRESH  = -0.02  # buy when IVol RoC < this (vol declining)

sig = pd.DataFrame(index=dbt.index)

# Signal 1: IVol regime gate (1 = safe to trade, 0 = blocked)
sig['ivol_gate'] = (dbt['ivol_pctl'] < IVOL_PCTL_GATE).astype(int)

# Signal 2: Calm vol regime (IVol pctl below entry threshold)
sig['calm_regime'] = (dbt['ivol_pctl'] < IVOL_PCTL_ENTRY).astype(int)

# Signal 3: Vol spread crossover (buy signal when spread turns negative \u2192 mean reversion)
sig['spread_cross_down'] = ((dbt['vol_spread'] < SPREAD_CROSS) &
                             (dbt['vol_spread'].shift(1) >= SPREAD_CROSS)).astype(int)
sig['spread_negative'] = (dbt['vol_spread'] < 0).astype(int)

# Signal 4: IVol declining momentum
sig['ivol_declining'] = (dbt['ivol_roc5'] < IVOL_ROC_THRESH).astype(int)

# Signal 5: IVol below its EMA (short-term vol compression)
sig['ivol_below_ema'] = (dbt['ivol'] < dbt['ivol_ewma20']).astype(int)

# Signal 6: SMA momentum (price above SMA20 and SMA20 above SMA50)
sig['trend_up'] = ((dbt['close'] > dbt['sma20']) & (dbt['sma20'] > dbt['sma50'])).astype(int)

# Composite: combine vol + trend signals
sig['composite_entry'] = (sig['ivol_gate'] & sig['calm_regime'] &
                          sig['ivol_below_ema'] & sig['trend_up']).astype(int)

# Show signal activity
print("Signal activity (% of days with signal = 1):")
for c in sig.columns:
    pct = sig[c].sum() / len(sig) * 100
    print(f"  {c:25s}  {pct:5.1f}%  ({sig[c].sum()} days)")
print(f"\nTotal days: {len(sig)}")

Signal activity (% of days with signal = 1):
  ivol_gate                   79.2%  (2271 days)
  calm_regime                 55.9%  (1603 days)
  spread_cross_down            0.8%  (23 days)
  spread_negative              4.4%  (126 days)
  ivol_declining              43.0%  (1235 days)
  ivol_below_ema              59.8%  (1717 days)
  trend_up                    38.7%  (1111 days)
  composite_entry             27.4%  (786 days)

Total days: 2869


## 5. Neural Network Signal Generation

Run all days through the TKAN model (single forward pass for speed).  
Flag days where the 10-day forecast predicts upside above our threshold.

In [15]:
# ── 5. NN prediction for every day in backtest period ────────────────
ENTRY_THRESHOLD = 1.0  # EUR above anchor to flag as bullish

nn_signals = pd.DataFrame(index=dates[bt_start_idx:], columns=[
    'nn_bullish', 'pred_max', 'pred_mean', 'pred_spread', 'cycle_key'])
nn_signals = nn_signals.astype({'nn_bullish': 'float64', 'pred_max': 'float64',
                                 'pred_mean': 'float64', 'pred_spread': 'float64'})

cycle_idx = 0
current_cycle_key = None

for count, i in enumerate(range(bt_start_idx, len(dates))):
    if cycle_idx < len(cycle_boundaries) and i >= cycle_boundaries[cycle_idx][0]:
        current_cycle_key = cycle_boundaries[cycle_idx][1]
        cycle_idx += 1

    if current_cycle_key is None or i < WINDOW_SIZE:
        continue

    if current_cycle_key not in model_cache:
        model_cache[current_cycle_key] = load_cycle(current_cycle_key, n_features)
    mdl, scl = model_cache[current_cycle_key]

    anchor = float(y.iloc[i])
    pred_prices = predict_single(mdl, scl, X, i, anchor)

    dt = dates[i]
    nn_signals.loc[dt, 'pred_max'] = float(pred_prices.max())
    nn_signals.loc[dt, 'pred_mean'] = float(pred_prices.mean())
    nn_signals.loc[dt, 'pred_spread'] = float(pred_prices.max() - pred_prices.min())
    nn_signals.loc[dt, 'nn_bullish'] = 1.0 if np.any(pred_prices >= anchor + ENTRY_THRESHOLD) else 0.0
    nn_signals.loc[dt, 'cycle_key'] = current_cycle_key

    if (count + 1) % 500 == 0:
        print(f"  {count+1}/{len(dates) - bt_start_idx}")

nn_signals['pred_upside'] = nn_signals['pred_max'] - y.reindex(nn_signals.index)

print(f"\nNN signals generated: {len(nn_signals.dropna())} days")
print(f"Bullish days: {nn_signals['nn_bullish'].sum():.0f} ({nn_signals['nn_bullish'].mean()*100:.1f}%)")
print(f"Avg predicted upside: EUR {nn_signals['pred_upside'].mean():.2f}")
print(f"Avg pred spread: EUR {nn_signals['pred_spread'].mean():.2f}")

  500/2869
  1000/2869
  1500/2869
  2000/2869
  2500/2869

NN signals generated: 2868 days
Bullish days: 742 (25.9%)
Avg predicted upside: EUR 0.69
Avg pred spread: EUR 1.28


## 6. Signal Overlay Charts

Overlay **rule-based** and **neural network** signals on price + volatility panels.  
Green triangles = entry signals, red = exits/blocked. Shading shows active regimes.

In [ ]:
# ── 6a. NN Signal + IVol Gate overlay on price ───────────────────────
# Merge signals
nn_sig = nn_signals[['nn_bullish', 'pred_upside', 'pred_spread']].dropna()
combined = sig.join(nn_sig, how='inner')

# Combined NN + vol gate signal
combined['nn_gated'] = (combined['nn_bullish'] == 1) & (combined['ivol_gate'] == 1)
combined['nn_gated_composite'] = combined['nn_gated'] & (combined['trend_up'] == 1)

# Build signal column for Signum: 1 = buy (NN gated), -1 = blocked, 0 = neutral
sig_col = pd.Series(0, index=combined.index)
sig_col[combined['nn_gated']] = 1
sig_col[(combined['nn_bullish'] == 1) & (combined['ivol_gate'] == 0)] = -1
sig_chart_df = pd.DataFrame({
    'time': combined.index.strftime('%Y-%m-%d'),
    'open': dbt.loc[combined.index, 'open'].values,
    'high': dbt.loc[combined.index, 'high'].values,
    'low': dbt.loc[combined.index, 'low'].values,
    'close': dbt.loc[combined.index, 'close'].values,
    'signal': sig_col.values,
})

# Pane 1: LVC candlestick + NN signals
p_s1 = Chart(height=350, watermark='LVC + Signals')
p_s1.candlestick(sig_chart_df)
p_s1.signals(sig_chart_df, signal_col='signal',
    buy_text='NN BUY', sell_text='BLOCKED', buy_color='#00AA00', sell_color='#CC0000')

# Shade high-vol periods on price
shade_sig = pd.DataFrame({
    'time': combined.index.strftime('%Y-%m-%d'),
    'position': (1 - combined['ivol_gate']).values
})
p_s1.shade(shade_sig, color='#FF0000', opacity=0.08)

# Pane 2: Predicted upside
p_s2 = Chart(height=180, watermark='Pred Upside')
p_s2.baseline(ts(combined['pred_upside'], 'value'), base_value=0)
p_s2.price_line(ENTRY_THRESHOLD, title=f'+{ENTRY_THRESHOLD} EUR', color='#FF0000')

# Pane 3: IVol percentile
p_s3 = Chart(height=180, watermark='IVol Pctl')
p_s3.line(ts(dbt['ivol_pctl'], 'value'), name='IVol Pctl', color='#FF8C00')
p_s3.price_line(IVOL_PCTL_GATE, title='Gate 80%', color='#FF0000')
p_s3.price_line(IVOL_PCTL_ENTRY, title='Entry 50%', color='#228B22')

# Pane 4: Composite signal as area
p_s4 = Chart(height=150, watermark='Composite Signal')
p_s4.area(ts(combined['nn_gated_composite'].astype(float), 'value'),
    name='NN+IVol+Trend', color='#228B22')

Dashboard(panes=[p_s1, p_s2, p_s3, p_s4],
    titles=['LVC Price + Signals', 'NN Predicted Upside (EUR)',
            'IVol Percentile + Gate', 'Composite Signal']).show()

In [ ]:
# ── 6b. NN prediction quality by vol regime ──────────────────────────
# How good are NN predictions when IVol is high vs low?
nn_merged = nn_signals.copy()
nn_merged['ivol_pctl'] = dbt['ivol_pctl'].reindex(nn_merged.index)
nn_merged['ivol'] = dbt['ivol'].reindex(nn_merged.index)
nn_merged['actual_return_10d'] = np.log(y.shift(-10) / y).reindex(nn_merged.index)
nn_merged = nn_merged.dropna()

# Define vol regimes
nn_merged['vol_regime'] = pd.cut(nn_merged['ivol_pctl'],
    bins=[0, 0.2, 0.5, 0.8, 1.0], labels=['Very Low', 'Low', 'Medium', 'High'])

# Scatter: predicted upside vs actual 10-day return, colored by vol regime
fig_qual = px.scatter(nn_merged.reset_index(), x='pred_upside', y='actual_return_10d',
    color='vol_regime', opacity=0.3, title='NN Prediction Quality by Vol Regime',
    labels={'pred_upside': 'NN Predicted Upside (EUR)', 'actual_return_10d': 'Actual 10d Log Return'},
    color_discrete_map={'Very Low': 'green', 'Low': 'lightgreen', 'Medium': 'orange', 'High': 'red'},
    height=500, width=1000)
fig_qual.add_hline(y=0, line_dash='dot', line_color='gray')
fig_qual.add_vline(x=0, line_dash='dot', line_color='gray')
fig_qual.update_layout(template='plotly_white')
fig_qual.show()

# Summary table by regime
regime_stats = nn_merged.groupby('vol_regime', observed=True).agg(
    days=('pred_upside', 'count'),
    avg_pred_upside=('pred_upside', 'mean'),
    avg_actual_10d_ret=('actual_return_10d', 'mean'),
    pct_bullish=('nn_bullish', 'mean'),
    avg_ivol=('ivol', 'mean'),
).round(4)
print("\nNN Prediction Quality by Vol Regime:")
regime_stats

## 7. Signal-Based Backtest

Three strategies compared:
1. **NN only**: Buy on NN bullish signal, sell at +1 EUR or after 10 days
2. **NN + IVol gate**: Same but block entries and force-close during high vol
3. **NN + Composite**: NN bullish + IVol safe + trend up

In [ ]:
# ── 7. Multi-strategy backtest engine ────────────────────────────────
def run_signal_backtest(signal_series, entry_threshold=1.0, max_hold_days=10,
                        ivol_gate_series=None, label='Strategy'):
    """
    Backtest long-only strategy driven by a boolean signal series.
    - Enter when signal = True (buy at close)
    - Exit when: close >= entry + entry_threshold, or max_hold_days reached, or ivol_gate fires
    Returns trades DataFrame and daily equity Series.
    """
    trades = []
    position = None
    cash = float(INITIAL_CAPITAL)
    shares = 0
    daily_eq = pd.Series(dtype=float, index=signal_series.index)

    for dt in signal_series.index:
        if dt not in y.index:
            daily_eq.loc[dt] = cash
            continue
        close = float(y.loc[dt])

        # Check ivol gate force-close
        if ivol_gate_series is not None and position is not None:
            gate_val = ivol_gate_series.get(dt, 1)
            if gate_val == 0:  # blocked
                pnl = shares * (close - position['entry_price'])
                cash += shares * close
                shares = 0
                trades.append({**position, 'exit_date': dt, 'exit_price': close,
                    'profit': close - position['entry_price'],
                    'days_held': (dt - position['entry_date']).days,
                    'exit_reason': 'ivol_gate'})
                position = None

        # Check exit conditions
        if position is not None:
            held = (dt - position['entry_date']).days
            if close >= position['entry_price'] + entry_threshold:
                cash += shares * close
                shares = 0
                trades.append({**position, 'exit_date': dt, 'exit_price': close,
                    'profit': close - position['entry_price'],
                    'days_held': held, 'exit_reason': 'take_profit'})
                position = None
            elif held >= max_hold_days:
                cash += shares * close
                shares = 0
                trades.append({**position, 'exit_date': dt, 'exit_price': close,
                    'profit': close - position['entry_price'],
                    'days_held': held, 'exit_reason': 'timeout'})
                position = None

        # Check entry
        if position is None and signal_series.loc[dt]:
            shares = int(cash // close)
            if shares > 0:
                cash -= shares * close
                position = {'entry_date': dt, 'entry_price': close,
                            'target': close + entry_threshold}

        # Mark daily equity
        daily_eq.loc[dt] = cash + (shares * close if shares > 0 else 0)

    # Close open position
    if position is not None and len(y) > 0:
        last = float(y.iloc[-1])
        cash += shares * last
        trades.append({**position, 'exit_date': y.index[-1], 'exit_price': last,
            'profit': last - position['entry_price'],
            'days_held': (y.index[-1] - position['entry_date']).days,
            'exit_reason': 'end'})
        daily_eq.iloc[-1] = cash

    return pd.DataFrame(trades), daily_eq

# Run 3 strategies
# Strategy 1: NN only
sig_nn = combined['nn_bullish'] == 1
trades_nn, eq_nn = run_signal_backtest(sig_nn, label='NN Only')

# Strategy 2: NN + IVol gate
sig_nn_gated = combined['nn_gated']
ivol_gate = combined['ivol_gate']
trades_gated, eq_gated = run_signal_backtest(sig_nn_gated, ivol_gate_series=ivol_gate, label='NN + IVol Gate')

# Strategy 3: NN + Composite (trend + vol)
sig_composite = combined['nn_gated_composite']
trades_comp, eq_comp = run_signal_backtest(sig_composite, ivol_gate_series=ivol_gate, label='NN + Composite')

print(f"Strategy 1 (NN Only):       {len(trades_nn)} trades")
print(f"Strategy 2 (NN + IVol):     {len(trades_gated)} trades")
print(f"Strategy 3 (NN + Composite): {len(trades_comp)} trades")

## 8. Performance Metrics & Equity Curves

In [ ]:
# ── 8a. Compute performance metrics for all strategies ───────────────
def compute_metrics(trades_df, equity_series, label):
    n = len(trades_df)
    if n == 0:
        return {'strategy': label, 'trades': 0}
    wins = (trades_df['profit'] > 0).sum()
    final_eq = equity_series.dropna().iloc[-1]
    daily_ret = equity_series.pct_change().dropna()
    years = (equity_series.index[-1] - equity_series.index[0]).days / 365.25
    cummax = equity_series.cummax()
    dd = ((equity_series - cummax) / cummax)
    return {
        'strategy': label,
        'trades': n,
        'win_rate': f"{wins/n*100:.1f}%",
        'total_pnl': f"EUR {trades_df['profit'].sum():,.2f}",
        'avg_profit': f"EUR {trades_df['profit'].mean():,.2f}",
        'avg_hold_days': f"{trades_df['days_held'].mean():.1f}",
        'sharpe': f"{daily_ret.mean() / daily_ret.std() * np.sqrt(252):.2f}" if daily_ret.std() > 0 else 'N/A',
        'max_dd': f"{dd.min()*100:.1f}%",
        'cagr': f"{((final_eq/INITIAL_CAPITAL)**(1/years)-1)*100:.1f}%" if years > 0 else 'N/A',
        'final_equity': f"EUR {final_eq:,.0f}",
        'roi': f"{(final_eq/INITIAL_CAPITAL - 1)*100:+.1f}%",
    }

# Buy & hold benchmark
bnh_entry = float(y.iloc[bt_start_idx])
bnh_exit = float(y.iloc[-1])
bnh_shares = int(INITIAL_CAPITAL // bnh_entry)
bnh_final = INITIAL_CAPITAL + bnh_shares * (bnh_exit - bnh_entry)
bnh_years = (y.index[-1] - y.index[bt_start_idx]).days / 365.25

results = [
    compute_metrics(trades_nn, eq_nn, 'NN Only'),
    compute_metrics(trades_gated, eq_gated, 'NN + IVol Gate'),
    compute_metrics(trades_comp, eq_comp, 'NN + Composite'),
]

results_df = pd.DataFrame(results).set_index('strategy')
print(f"Buy & Hold: EUR {INITIAL_CAPITAL:,} \u2192 EUR {bnh_final:,.0f} "
      f"(ROI: {(bnh_final/INITIAL_CAPITAL-1)*100:+.1f}%, "
      f"CAGR: {((bnh_final/INITIAL_CAPITAL)**(1/bnh_years)-1)*100:.1f}%)")
print()
results_df

In [ ]:
# ── 8b. Equity curves comparison chart (Signum) ──────────────────────
# Pane 1: Equity curves
p_eq1 = Chart(height=350, watermark='Equity Comparison')
for eq_s, name, color in [
    (eq_nn, 'NN Only', '#228B22'),
    (eq_gated, 'NN + IVol Gate', '#800080'),
    (eq_comp, 'NN + Composite', '#DC143C'),
]:
    eq_clean = eq_s.dropna()
    p_eq1.line(ts(eq_clean, 'value'), name=name, color=color)

bnh_eq = INITIAL_CAPITAL + bnh_shares * (y.loc[dbt.index].reindex(eq_nn.dropna().index) - bnh_entry)
p_eq1.line(ts(bnh_eq, 'value'), name='Buy & Hold', color='#808080', width=1)
p_eq1.price_line(INITIAL_CAPITAL, title='Initial Capital', color='#808080')

# Pane 2: Drawdown
p_eq2 = Chart(height=200, watermark='Drawdown')
for eq_s, name, color in [
    (eq_nn, 'NN Only DD', '#228B22'),
    (eq_gated, 'NN + IVol Gate DD', '#800080'),
    (eq_comp, 'NN + Composite DD', '#DC143C'),
]:
    eq_clean = eq_s.dropna()
    cummax = eq_clean.cummax()
    dd = (eq_clean - cummax) / cummax * 100
    p_eq2.line(ts(dd, 'value'), name=name, color=color, width=1)
p_eq2.price_line(0.0, title='zero', color='#808080')

Dashboard(panes=[p_eq1, p_eq2],
    titles=['Equity Curves \u2014 Strategy Comparison', 'Drawdown (%)']).show()

## 9. Interactive Parameter Tuning

Adjust IVol gate threshold, entry threshold, and max hold days. Re-run the backtest with different params to see what works.

In [ ]:
# ── 9. Parameter sweep: IVol gate \u00d7 Entry threshold \u00d7 Max hold days ──
from itertools import product

# Grid \u2014 adjust these ranges to explore
ivol_gates   = [0.60, 0.70, 0.80, 0.90, 1.0]   # 1.0 = no gate
entry_threshs = [0.5, 1.0, 1.5, 2.0]             # EUR above current close
max_holds     = [5, 10, 15, 20]                    # max days to hold

sweep_results = []

for gate, thresh, hold in product(ivol_gates, entry_threshs, max_holds):
    # Build signal: NN bullish + IVol gate at this level
    gate_sig = (dbt['ivol_pctl'] < gate).reindex(combined.index).fillna(True)
    entry_sig = (combined['nn_bullish'] == 1) & gate_sig
    ivol_block = gate_sig.astype(int)

    tr, eq = run_signal_backtest(entry_sig, entry_threshold=thresh, max_hold_days=hold,
                                  ivol_gate_series=ivol_block, label='sweep')
    if len(tr) == 0:
        continue

    eq_clean = eq.dropna()
    if len(eq_clean) < 2:
        continue
    final = eq_clean.iloc[-1]
    daily_r = eq_clean.pct_change().dropna()
    sharpe = (daily_r.mean() / daily_r.std() * np.sqrt(252)) if daily_r.std() > 0 else 0
    cummax = eq_clean.cummax()
    max_dd = ((eq_clean - cummax) / cummax).min() * 100
    years = (eq_clean.index[-1] - eq_clean.index[0]).days / 365.25
    cagr = ((final / INITIAL_CAPITAL) ** (1 / max(years, 0.1)) - 1) * 100

    wins = (tr['profit'] > 0).sum()
    sweep_results.append({
        'ivol_gate': gate, 'entry_thresh': thresh, 'max_hold': hold,
        'trades': len(tr), 'win_rate': wins / len(tr) * 100,
        'sharpe': round(sharpe, 2), 'max_dd': round(max_dd, 1),
        'cagr': round(cagr, 1), 'roi': round((final / INITIAL_CAPITAL - 1) * 100, 1),
        'final_equity': round(final, 0),
    })

sweep_df = pd.DataFrame(sweep_results).sort_values('sharpe', ascending=False)
print(f"Parameter sweep: {len(sweep_df)} valid combos")
print(f"\nTop 15 by Sharpe:")
sweep_df.head(15)

In [ ]:
# ── 9b. Heatmap: Sharpe ratio by IVol gate \u00d7 Entry threshold ─────────
# Aggregate across max_hold (take best)
heatmap_data = sweep_df.groupby(['ivol_gate', 'entry_thresh'])['sharpe'].max().reset_index()
heatmap_pivot = heatmap_data.pivot(index='ivol_gate', columns='entry_thresh', values='sharpe')

fig_heat = go.Figure(data=go.Heatmap(
    z=heatmap_pivot.values, x=[f"EUR {t}" for t in heatmap_pivot.columns],
    y=[f"{g:.0%}" for g in heatmap_pivot.index],
    colorscale='RdYlGn', text=heatmap_pivot.values.round(2), texttemplate='%{text}',
    hovertemplate='IVol Gate: %{y}<br>Entry Threshold: %{x}<br>Sharpe: %{z:.2f}<extra></extra>'))

fig_heat.update_layout(title='Best Sharpe Ratio \u2014 IVol Gate \u00d7 Entry Threshold',
    xaxis_title='Entry Threshold', yaxis_title='IVol Gate Percentile',
    height=400, width=700, template='plotly_white')
fig_heat.show()

# CAGR heatmap
heatmap_cagr = sweep_df.groupby(['ivol_gate', 'entry_thresh'])['cagr'].max().reset_index()
heatmap_pivot_c = heatmap_cagr.pivot(index='ivol_gate', columns='entry_thresh', values='cagr')

fig_heat2 = go.Figure(data=go.Heatmap(
    z=heatmap_pivot_c.values, x=[f"EUR {t}" for t in heatmap_pivot_c.columns],
    y=[f"{g:.0%}" for g in heatmap_pivot_c.index],
    colorscale='RdYlGn', text=heatmap_pivot_c.values.round(1), texttemplate='%{text}%',
    hovertemplate='IVol Gate: %{y}<br>Entry Threshold: %{x}<br>CAGR: %{z:.1f}%<extra></extra>'))

fig_heat2.update_layout(title='Best CAGR \u2014 IVol Gate \u00d7 Entry Threshold',
    xaxis_title='Entry Threshold', yaxis_title='IVol Gate Percentile',
    height=400, width=700, template='plotly_white')
fig_heat2.show()

## 10. Scratch Pad

Free-form exploration. Try different ideas, overlay new indicators, test signal combinations.

In [ ]:
# ── 10a. Quick-run: re-backtest with custom params ───────────────────
# Edit these and re-run this cell to test any combination instantly

CUSTOM_GATE    = 0.75   # IVol percentile gate
CUSTOM_THRESH  = 1.0    # EUR entry threshold
CUSTOM_HOLD    = 10     # Max hold days
USE_TREND      = True   # Require trend_up signal too?

# Build signal
gate_s = (dbt['ivol_pctl'] < CUSTOM_GATE).reindex(combined.index).fillna(True)
entry_s = (combined['nn_bullish'] == 1) & gate_s
if USE_TREND:
    entry_s = entry_s & (combined['trend_up'] == 1)
ivol_g = gate_s.astype(int)

tr_custom, eq_custom = run_signal_backtest(entry_s, entry_threshold=CUSTOM_THRESH,
    max_hold_days=CUSTOM_HOLD, ivol_gate_series=ivol_g, label='Custom')

# Quick metrics
if len(tr_custom) > 0:
    eq_c = eq_custom.dropna()
    final_c = eq_c.iloc[-1]
    dr = eq_c.pct_change().dropna()
    sh = dr.mean() / dr.std() * np.sqrt(252) if dr.std() > 0 else 0
    cum = eq_c.cummax()
    mdd = ((eq_c - cum) / cum).min() * 100
    yrs = (eq_c.index[-1] - eq_c.index[0]).days / 365.25
    cagr_c = ((final_c / INITIAL_CAPITAL) ** (1 / max(yrs, 0.1)) - 1) * 100
    wins_c = (tr_custom['profit'] > 0).sum()

    print(f"Custom Strategy: gate={CUSTOM_GATE:.0%}, thresh=EUR {CUSTOM_THRESH}, hold={CUSTOM_HOLD}d, trend={USE_TREND}")
    print(f"Trades: {len(tr_custom)}  Win rate: {wins_c/len(tr_custom)*100:.1f}%")
    print(f"Sharpe: {sh:.2f}  Max DD: {mdd:.1f}%  CAGR: {cagr_c:.1f}%")
    print(f"EUR {INITIAL_CAPITAL:,} \u2192 EUR {final_c:,.0f} (ROI: {(final_c/INITIAL_CAPITAL-1)*100:+.1f}%)")

    # Build signal column for Signum markers
    trade_sig = pd.Series(0, index=combined.index, dtype=int)
    for _, t in tr_custom.iterrows():
        if t['entry_date'] in trade_sig.index:
            trade_sig.loc[t['entry_date']] = 1
        if t['exit_date'] in trade_sig.index:
            trade_sig.loc[t['exit_date']] = -1
    trade_chart_df = pd.DataFrame({
        'time': dbt.index.strftime('%Y-%m-%d'),
        'open': dbt['open'].values, 'high': dbt['high'].values,
        'low': dbt['low'].values, 'close': dbt['close'].values,
    })
    sig_df = pd.DataFrame({
        'time': combined.index.strftime('%Y-%m-%d'),
        'signal': trade_sig.values,
    })

    # Pane 1: LVC candlestick + buy/sell markers
    p_c1 = Chart(height=350, watermark=f'Custom: gate={CUSTOM_GATE:.0%} thresh=EUR {CUSTOM_THRESH}')
    p_c1.candlestick(trade_chart_df)
    p_c1.signals(sig_df, buy_text='BUY', sell_text='SELL')

    # Shade positions
    pos_df = pd.DataFrame({'time': combined.index.strftime('%Y-%m-%d'), 'position': 0})
    for _, t in tr_custom.iterrows():
        mask = (combined.index >= t['entry_date']) & (combined.index <= t['exit_date'])
        pos_df.loc[mask.values, 'position'] = 1
    p_c1.shade(pos_df, color='#00AA00', opacity=0.06)

    # Pane 2: Equity curve
    p_c2 = Chart(height=250, watermark='Equity')
    p_c2.line(ts(eq_c, 'value'), name='Custom Equity', color='#800080')
    bnh_c = INITIAL_CAPITAL + bnh_shares * (y.reindex(eq_c.index) - bnh_entry)
    p_c2.line(ts(bnh_c, 'value'), name='Buy & Hold', color='#808080', width=1)
    p_c2.price_line(INITIAL_CAPITAL, title='Initial', color='#808080')

    Dashboard(panes=[p_c1, p_c2],
        titles=[f'LVC + Custom Signals | Sharpe={sh:.2f} CAGR={cagr_c:.1f}%',
                'Custom Equity vs Buy & Hold']).show()
else:
    print("No trades generated with these parameters.")

In [ ]:
# ── 10b. Vol regime scatter: LVC 10d forward return vs IVol features ─
# Useful for eyeballing vol levels that predict good/bad returns
fwd = np.log(y.shift(-10) / y).reindex(dbt.index).rename('fwd_10d')
scatter_df = dbt[['ivol', 'ivol_pctl', 'vol_spread', 'rvol_park20', 'ivol_zscore']].join(fwd).dropna()

fig_scatter = make_subplots(rows=2, cols=2, horizontal_spacing=0.08, vertical_spacing=0.10,
    subplot_titles=("IVol Level vs 10d Fwd Return", "IVol Percentile vs 10d Fwd Return",
                    "Vol Spread vs 10d Fwd Return", "IVol Z-Score vs 10d Fwd Return"))

for i, (col, color) in enumerate([('ivol', 'orange'), ('ivol_pctl', 'darkorange'),
                                    ('vol_spread', 'purple'), ('ivol_zscore', 'crimson')]):
    r, c = divmod(i, 2)
    fig_scatter.add_trace(go.Scatter(x=scatter_df[col], y=scatter_df['fwd_10d'],
        mode='markers', marker=dict(size=3, color=color, opacity=0.3),
        name=col, showlegend=False), row=r+1, col=c+1)
    # Add regression line
    mask = scatter_df[[col, 'fwd_10d']].dropna()
    if len(mask) > 10:
        z = np.polyfit(mask[col], mask['fwd_10d'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(mask[col].min(), mask[col].max(), 100)
        fig_scatter.add_trace(go.Scatter(x=x_line, y=p(x_line),
            mode='lines', line=dict(color='black', width=2, dash='dash'),
            name=f'trend', showlegend=False), row=r+1, col=c+1)
    fig_scatter.add_hline(y=0, line_dash='dot', line_color='gray', opacity=0.3, row=r+1, col=c+1)

fig_scatter.update_layout(height=700, width=1200, template='plotly_white',
    title='Volatility Features vs 10-Day Forward Return')
fig_scatter.show()